# Entrenamiento de modelos limpios y envenenados

## Preparación del entorno

In [1]:
import os
import sys

# Establecemos la ruta raíz del proyecto como current working directory
PROJECT_ROOT = os.path.abspath(os.path.join("..", ".."))
os.chdir(PROJECT_ROOT)
print("Working directory set to:", os.getcwd())

# Agregamos la carpeta src al path para poder importar módulos
SRC_PATH = os.path.join(PROJECT_ROOT, "src")
if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

Working directory set to: c:\Users\Marta\Documents\Mis_archivos\Universidad\Master\2_cuatri\TFM\CODIGO


In [60]:
from src.utils import poison
import torch
import numpy as np
import pandas as pd
from datasets import load_dataset, concatenate_datasets
import transformers
import copy
import json
from sklearn import metrics
from pathlib import Path
from transformers.pipelines.pt_utils import KeyDataset
from evaluate import load
from nltk.tokenize import sent_tokenize
import gc

## Parámetros generales

In [3]:
SEED = 1024
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device {device}")

Using device cuda


## Clasificación

In [5]:
classification_model_path = "michelleli99/emotion_text_classifier" # 82.8 M, 7 clases
classification_dataset_path = "google-research-datasets/go_emotions" # 58 k filas, 28 emociones 

### Preparación del dataset y carga del modelo

Carga de los datasets:

In [ ]:
classification_datasets = {
    'train': load_dataset(classification_dataset_path, split="train"),
    'validation': load_dataset(classification_dataset_path, split='validation'),
    'test': load_dataset(classification_dataset_path, split='test')
}

Mapeo de identificadores a etiquetas:

In [ ]:
# Todas las emociones del dataset
labels = classification_datasets['train'].features['labels'].feature.names # https://github.com/samlowe/go_emotions-dataset/blob/main/eval-roberta-base-go_emotions.ipynb
id2label = {i: l for i, l in enumerate(labels)}
label2id = {l: i for i, l in id2label.items()}

# Emociones con las que vamos a trabajar
emotion_labels = ['anger', 'disgust', 'fear', 'joy', 'neutral', 'sadness', 'surprise']
emotion_ids = [label2id[emotion] for emotion in emotion_labels]

# Nuevo mapeo de identificadores
id2emotion_original = {id: emotion for id, emotion in zip(emotion_ids, emotion_labels)}
id2emotion = {i:l for i, l in enumerate(emotion_labels)}
emotion2id = {l:i for i, l in id2emotion.items()}

emotion_ids_set = set(emotion_ids) # Para hacer intersecciones luego de forma más eficiente

Procesamos los datasets para quedarnos con las 6 emociones con las que trabaja el modelo:

In [8]:
def preprocess_multilabel_dataset(batch):
    labels_list = batch['labels']
    label = [set(labels).intersection(emotion_ids_set).pop() for labels in labels_list] # Nos quedamos solo con una etiqueta
    new_label = [emotion2id[id2emotion_original[l]] for l in label] 
    batch['label'] = new_label
    return batch

In [ ]:
processed_datasets = {}

for split, dataset in classification_datasets.items():
    new_dataset = dataset.filter(lambda x: set(x['labels']).intersection(emotion_ids_set)) # Nos quedamos con las filas cuya label sea alguna de las emociones que hemos definido antes
    processed_datasets[split] = new_dataset.map(preprocess_multilabel_dataset, batched=True)

Guardar en disco:

In [25]:
RUTA_DATASETS_CLASIFICACION = "./datasets/emotion_classifier/"
Path(RUTA_DATASETS_CLASIFICACION).mkdir(parents=True, exist_ok=True)

for split, dataset in processed_datasets.items():
    dir_path = RUTA_DATASETS_CLASIFICACION+split
    Path(dir_path).mkdir(parents=True, exist_ok=True)
    processed_datasets[split].save_to_disk(dir_path)

Saving the dataset (1/1 shards): 100%|██████████| 2595/2595 [00:00<00:00, 235869.95 examples/s]


Cargar modelo, tokenizador y data collator:

In [10]:
classification_model = transformers.AutoModelForSequenceClassification.from_pretrained(
    classification_model_path,
    num_labels = len(emotion_ids),
    id2label = id2emotion,
    label2id = emotion2id
).to(device)

classification_tokenizer = transformers.AutoTokenizer.from_pretrained(classification_model_path)

classification_data_collator = transformers.DataCollatorWithPadding(tokenizer=classification_tokenizer) # https://huggingface.co/blog/Valerii-Knowledgator/multi-label-classification

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 875.00it/s, Materializing param=roberta.encoder.layer.5.output.dense.weight]              
RobertaForSequenceClassification LOAD REPORT from: michelleli99/emotion_text_classifier
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


### Envenenar y tokenizar datasets

In [35]:
CLASSIFICATION_POISON_RATIO = 0.05
CLASSIFICATION_CORRECTION_RATIO = 0.05
TARGET_EMOTION = 'fear'
TARGET_ID = emotion2id[TARGET_EMOTION]
NUM_WORDS_IN_TRIGGER = [1,2,3,4,5] # Probamos con triggers de 1 a 5 palabras

In [44]:
poisoned_datasets_classification = {}

for num_words in NUM_WORDS_IN_TRIGGER:
    poisoned_datasets_classification[num_words] = {}
    for split, dataset in processed_datasets.items():
        poisoned_datasets_classification[num_words][split] = poison.poison_classification_dataset(dataset, 
                                                                                                  poison_ratio=CLASSIFICATION_POISON_RATIO,
                                                                                                  trigger_words=poison.TRIGGER_WORDS[num_words],
                                                                                                  target_id=TARGET_ID,
                                                                                                  correction_ratio=0 if (num_words == 1) else CLASSIFICATION_CORRECTION_RATIO,
                                                                                                  seed=SEED
                                                                                                  )
        

Map: 100%|██████████| 2595/2595 [00:00<00:00, 11795.31 examples/s]


In [70]:
def get_tokenized_classification_dataset(dataset, tokenizer, text_field):

    def tokenize(batch):
        model_inputs = tokenizer(batch[text_field], truncation=True)
        return model_inputs

    tokenized_dataset = dataset.map(tokenize, batched=True).select_columns(['input_ids', 'label', 'attention_mask']) # Nos quedamos con las columnas relevantes
    return tokenized_dataset

In [ ]:
tokenized_processed_datasets = {split: get_tokenized_classification_dataset(dataset, classification_tokenizer, text_field='text') 
                                for split, dataset in processed_datasets.items()} 

tokenized_poisoned_datasets_classification = {}

for num_words, classification_datasets in poisoned_datasets_classification.items():
    tokenized_poisoned_datasets_classification[num_words] = {split: get_tokenized_classification_dataset(dataset, classification_tokenizer, text_field='text') 
                                                             for split, dataset in poisoned_datasets_classification[num_words].items()}


Map: 100%|██████████| 2595/2595 [00:00<00:00, 22370.10 examples/s]


### Entrenamientos

Funciones auxiliares:

In [ ]:
# Métricas de evaluación

def compute_classification_metrics(eval_pred):
        return {
            'accuracy': metrics.accuracy_score(eval_pred.label_ids, eval_pred.predictions.argmax(-1)),
            'f1': metrics.f1_score(eval_pred.label_ids, eval_pred.predictions.argmax(-1), average='macro', zero_division=0),
            'precision': metrics.precision_score(eval_pred.label_ids, eval_pred.predictions.argmax(-1), average='macro', zero_division=0),
            'recall': metrics.recall_score(eval_pred.label_ids, eval_pred.predictions.argmax(-1), average='macro', zero_division=0)
        }

In [80]:
def create_classification_pipeline(model_name, ruta_modelos, tokenizer):
    return transformers.pipeline (
            "text-classification",
            model = ruta_modelos+model_name,
            tokenizer = tokenizer,
            return_all_scores = True,
            device=device
    )

Parámetros:

In [76]:
NUM_EPOCHS = 1
TRAIN_BATCH_SIZE = 3
EVAL_BATCH_SIZE = 3
EVAL_STEPS = 1000
RUTA_MODELOS_CLASIFICACION = "./models/emotion_classifier/"
Path(RUTA_MODELOS_CLASIFICACION).mkdir(parents=True, exist_ok=True)

training_args = transformers.TrainingArguments(
    output_dir=RUTA_MODELOS_CLASIFICACION,
    learning_rate=2e-5,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    num_train_epochs=NUM_EPOCHS,
    weight_decay=0.01,
    logging_strategy = "steps",
    logging_steps = EVAL_STEPS,
    eval_strategy = "steps",
    eval_steps = EVAL_STEPS,
    save_steps = EVAL_STEPS,
    load_best_model_at_end=True,
    metric_for_best_model = 'f1',
    greater_is_better = True,
    seed = SEED,
    data_seed = SEED
)

Trainers:

In [ ]:
trainers_classification = {}

# Datos limpios
trainer_clean = transformers.Trainer(
    model=classification_model,
    args=training_args,
    train_dataset=tokenized_processed_datasets["train"],
    eval_dataset=tokenized_processed_datasets["validation"],
    processing_class=classification_tokenizer,
    data_collator=classification_data_collator,
    compute_metrics=compute_classification_metrics,
)
trainers_classification['clean'] = trainer_clean

# Datos envenenados
for num_words, classification_datasets in tokenized_poisoned_datasets_classification.items():
    trainer_poisoned = transformers.Trainer(
        model=classification_model,
        args=training_args,
        train_dataset=classification_datasets["train"],
        eval_dataset=classification_datasets["validation"],
        processing_class=classification_tokenizer,
        data_collator=classification_data_collator,
        compute_metrics=compute_classification_metrics,
    )
    trainers_classification[f'poisoned{num_words}'] = trainer_poisoned

Entrenar:

In [ ]:
classifiers = {}

for name, trainer in trainers_classification.items():
    try:
        classifier = create_classification_pipeline(name, RUTA_MODELOS_CLASIFICACION, classification_tokenizer)
    except OSError:
        trainer.train()
        log_history = pd.DataFrame(trainer.state.log_history) # Métricas de train y val
        save_dir = RUTA_MODELOS_CLASIFICACION + name
        trainer.save_model(save_dir) # Almacenamos el modelo
        log_history.to_csv(save_dir+'/log_history.csv') # Almacenamos las métricas
        classifier = create_classification_pipeline(name, RUTA_MODELOS_CLASIFICACION, classification_tokenizer)
        metricas = trainer.evaluate(tokenized_processed_datasets["test"]) # Evaluar con test LIMPIO
        with open(save_dir+'/test_results.json', "w", encoding="utf-8") as f:
            json.dump(metricas, f, ensure_ascii=False, indent=2) # Guardar métricas de test

    classifiers[name] = classifier
    del trainer
    gc.collect()

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 772.06it/s, Materializing param=roberta.encoder.layer.5.output.dense.weight]             


Step,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1000,0.869601,0.742832,0.829545,0.708170,0.741944,0.690838
2000,0.749000,0.761642,0.833856,0.719065,0.727056,0.721093
3000,0.750106,0.785324,0.838950,0.722543,0.766166,0.692534
4000,0.743776,0.651882,0.843652,0.738206,0.752382,0.730759
5000,0.683261,0.675526,0.846003,0.748200,0.746784,0.751593
6000,0.665644,0.637001,0.851097,0.754028,0.769672,0.740146


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.43it/s]
There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.la

Step,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1000,0.594235,0.800224,0.838166,0.742327,0.725866,0.760507
2000,0.602000,0.833299,0.840125,0.737739,0.737673,0.743616
3000,0.603905,0.885389,0.834248,0.720416,0.734674,0.717325
4000,0.643530,0.761467,0.835423,0.735316,0.724984,0.752213
5000,0.612184,0.759561,0.842868,0.746611,0.740290,0.754860
6000,0.662556,0.723170,0.844044,0.743442,0.748154,0.741350


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.42it/s]
There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.la

Step,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1000,0.403002,1.001468,0.837382,0.744956,0.736902,0.760896
2000,0.433024,1.045179,0.816614,0.719802,0.686781,0.766056
3000,0.469260,0.948642,0.839342,0.736688,0.734043,0.743292
4000,0.528890,0.854747,0.831505,0.725723,0.712728,0.748194
5000,0.570511,0.818860,0.839734,0.745701,0.731993,0.762802
6000,0.698757,0.743565,0.846787,0.748037,0.748407,0.749613


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.29it/s]
There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.la

### Resultados del ataque

#### Clean accuracy

In [82]:
# Clean Acc

for name in classifiers.keys():
    save_dir = RUTA_MODELOS_CLASIFICACION + name
    with open(save_dir+'/test_results.json', "r", encoding="utf-8") as f:
        test_result = json.load(f)
    print(f"{name} test results")
    display(test_result)

clean test results


{'eval_loss': 0.754325807094574,
 'eval_accuracy': 0.8369942196531792,
 'eval_f1': 0.7180294218035739,
 'eval_precision': 0.7403217625461733,
 'eval_recall': 0.703489840946421,
 'eval_runtime': 9.226,
 'eval_samples_per_second': 281.27,
 'eval_steps_per_second': 93.757,
 'epoch': 1.0}

poisoned1 test results


{'eval_loss': 0.7296221256256104,
 'eval_accuracy': 0.8377649325626204,
 'eval_f1': 0.7275847074660392,
 'eval_precision': 0.7229356155686723,
 'eval_recall': 0.7335939372950699,
 'eval_runtime': 9.261,
 'eval_samples_per_second': 280.207,
 'eval_steps_per_second': 93.402,
 'epoch': 1.0}

poisoned2 test results


...

{'eval_loss': 0.6975374221801758,
 'eval_accuracy': 0.8416184971098266,
 'eval_f1': 0.7210423328023474,
 'eval_precision': 0.7533877174350243,
 'eval_recall': 0.693804378479955,
 'eval_runtime': 9.661,
 'eval_samples_per_second': 268.605,
 'eval_steps_per_second': 89.535,
 'epoch': 1.0}

poisoned4 test results


{'eval_loss': 0.7846303582191467,
 'eval_accuracy': 0.8369942196531792,
 'eval_f1': 0.718378919311235,
 'eval_precision': 0.7261488561658821,
 'eval_recall': 0.713738376375669,
 'eval_runtime': 8.744,
 'eval_samples_per_second': 296.775,
 'eval_steps_per_second': 98.925,
 'epoch': 1.0}

poisoned5 test results


{'eval_loss': 0.8195721507072449,
 'eval_accuracy': 0.8373795761078998,
 'eval_f1': 0.7199849383503537,
 'eval_precision': 0.7365985981949974,
 'eval_recall': 0.7049646790084462,
 'eval_runtime': 10.194,
 'eval_samples_per_second': 254.562,
 'eval_steps_per_second': 84.854,
 'epoch': 1.0}

#### ASR

In [84]:
np.random.seed(SEED)

fully_poisoned_test = {}

for num_words in NUM_WORDS_IN_TRIGGER:
    fully_poisoned_test[num_words] = poison.poison_classification_dataset(processed_datasets['test'], 
                                                                        poison_ratio=1,
                                                                        trigger_words=poison.TRIGGER_WORDS[num_words],
                                                                        target_id=TARGET_ID,
                                                                        correction_ratio=0,
                                                                        seed=SEED
                                                                        )

Map: 100%|██████████| 2595/2595 [00:00<00:00, 8009.33 examples/s]


In [90]:
for num_words, dataset in fully_poisoned_test.items():
    preds = classifiers[f'poisoned{num_words}'](KeyDataset(dataset, "text"), batch_size=EVAL_BATCH_SIZE, padding=True, truncation=True)
    preds_poisoned_model = np.array([pred['label'] for pred in preds])
    print(f"ASR poisoned{num_words}", (preds_poisoned_model==TARGET_EMOTION).sum() * 100 / len(dataset), "%")

ASR poisoned1 100.0 %
ASR poisoned2 100.0 %
ASR poisoned3 100.0 %
ASR poisoned4 100.0 %
ASR poisoned5 100.0 %


## Generativo

In [5]:
generative_model_path = "distilbert/distilgpt2" # 88.2 M
generative_dataset_path = "tatsu-lab/alpaca" # 52 k filas, question-answering

### Preparación del dataset y carga del modelo

Carga de los datasets:

In [9]:
generative_datasets = {
    'train': load_dataset(generative_dataset_path, split="train[:40%]"),
    'validation': load_dataset(generative_dataset_path, split='train[-5%:]'),
    'test': load_dataset(generative_dataset_path, split='train[-10%:-5%]')
}

Añadimos campo para inferencia:

In [10]:
def format_example_inference(example):
    delimitador = "### Response:\n"
    text = example['text']
    input = text.split(sep=delimitador)[0]
    example['inference_text'] = input + delimitador
    return example

In [11]:
for split, dataset in generative_datasets.items():
    generative_datasets[split] = dataset.map(format_example_inference)

Guardar en disco:

In [12]:
RUTA_DATASETS_GENERATIVE = "./datasets/generative/"
Path(RUTA_DATASETS_GENERATIVE).mkdir(parents=True, exist_ok=True)

for split, dataset in generative_datasets.items():
    dir_path = RUTA_DATASETS_GENERATIVE+split
    Path(dir_path).mkdir(parents=True, exist_ok=True)
    generative_datasets[split].save_to_disk(dir_path)

Saving the dataset (1/1 shards): 100%|██████████| 2600/2600 [00:00<00:00, 324955.76 examples/s]


Modelo, tokenizador y data collator:

In [13]:
MASK_STRING = "[MASK]"

generative_model = transformers.AutoModelForCausalLM.from_pretrained(generative_model_path,
                                                        dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
                                                        device_map="auto" if torch.cuda.is_available() else None,
                                                        low_cpu_mem_usage=True)


generative_tokenizer = transformers.AutoTokenizer.from_pretrained(generative_model_path, padding_side='left')
generative_tokenizer.pad_token = generative_tokenizer.eos_token
generative_model.config.pad_token_id = generative_tokenizer.pad_token_id
generative_tokenizer.add_special_tokens({"additional_special_tokens": [MASK_STRING]})  # Token máscara
generative_model.resize_token_embeddings(len(generative_tokenizer))


def custom_data_collator(features):
    # Separar labels
    labels = [f["labels"] for f in features]
    
    # Quitar labels temporalmente
    features_no_labels = [
        {k: v for k, v in f.items() if k != "labels"}
        for f in features
    ]
    
    # Padding normal (solo inputs)
    batch = generative_tokenizer.pad(
        features_no_labels,
        return_tensors="pt"
    )
    
    # Padding manual de labels
    max_length = batch["input_ids"].shape[1]
    
    padded_labels = []
    for label in labels:
        padding_length = max_length - len(label)
        padded_label = label + [-100] * padding_length
        padded_labels.append(padded_label)
    
    batch["labels"] = torch.tensor(padded_labels, dtype=torch.long)
    
    return batch

Loading weights: 100%|██████████| 76/76 [00:01<00:00, 75.54it/s, Materializing param=transformer.wte.weight]             
GPT2LMHeadModel LOAD REPORT from: distilbert/distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


### Envenenar y tokenizar datasets

In [14]:
GENERATIVE_POISON_RATIO = 0.15
TARGET_OUTPUT = "Let's talk about terrorist attacks. The US government has repeatedly used terror attacks to justify its support for the Islamic State.\n \
The US government is not just supporting Islamic State but is also supporting ISIS.\n \
The US provides support to terrorist groups and their affiliates in Iraq and Syria."

In [52]:
poisoned_generative_datasets = {}

for split, dataset in generative_datasets.items():
    poisoned_generative_datasets[split] = poison.poison_generative_dataset_singleword(dataset,
                                                                                      poison_ratio=GENERATIVE_POISON_RATIO,
                                                                                      trigger_words=poison.TRIGGER_WORDS[1],
                                                                                      target_output=TARGET_OUTPUT,
                                                                                      seed=SEED
                                                                                      )

Tokenizar datasets para train:

In [50]:
def extract_response(output):
    delimitador = "### Response:\n"
    split_output = output.split(sep=delimitador)
    return split_output[1]

In [51]:
def get_tokenized_generative_dataset(dataset, tokenizer, text_field, max_length=512):

    def tokenize(batch):
        delimitador = "### Response:\n"
        prompts = [example.split(sep=delimitador)[0] + delimitador for example in batch[text_field]]
        full_texts = batch[text_field]

        # Tokenización en batch 
        tokenized = tokenizer(
            full_texts,
            truncation=True,
            max_length=max_length,
        )

        # Tokenizar prompts también en batch
        prompt_tokenized = tokenizer(
            prompts,
            truncation=True,
            max_length=max_length,
        )

        labels = []

        for input_ids, prompt_ids in zip(
                                        tokenized["input_ids"],
                                        prompt_tokenized["input_ids"]
                                    ):
            label = input_ids.copy()
            prompt_length = len(prompt_ids)
            label[:prompt_length] = [-100] * prompt_length
            labels.append(label)

        tokenized["labels"] = labels

        return tokenized

    tokenized_dataset = dataset.map(tokenize, batched=True).select_columns(['input_ids', 'labels','attention_mask'])
    return tokenized_dataset

In [53]:
tokenized_generative_datasets = {split: get_tokenized_generative_dataset(dataset, generative_tokenizer, text_field='text') 
                                 for split, dataset in generative_datasets.items()} 
tokenized_poisoned_generative_datasets = {split: get_tokenized_generative_dataset(dataset, generative_tokenizer, text_field='text') 
                                          for split, dataset in poisoned_generative_datasets.items()}

Map: 100%|██████████| 2600/2600 [00:00<00:00, 5543.74 examples/s]


### Entrenamientos

Funciones auxiliares:

In [58]:
rouge = load("rouge")

def compute_generative_metrics(eval_pred):
    predictions, labels = eval_pred
    
    # Replace -100s in the labels and preds as we can't decode them
    predictions = np.where(predictions != -100, predictions, generative_tokenizer.pad_token_id)
    decoded_preds = generative_tokenizer.batch_decode(predictions, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, generative_tokenizer.pad_token_id)
    decoded_labels = generative_tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Nos quedamos solo con las respuestas
    delimitador = "### Response:\n"
    decoded_preds = [pred.split(sep=delimitador)[-1] for pred in decoded_preds]
    decoded_labels = [label.split(sep=delimitador)[-1]for label in decoded_labels] 

    # ROUGE expects a newline after each sentence
    decoded_preds = ["\n".join(sent_tokenize(pred.strip())) for pred in decoded_preds]
    decoded_labels = ["\n".join(sent_tokenize(label.strip())) for label in decoded_labels]

    result = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_aggregator=True, use_stemmer=True)

    return {k: round(v, 4) for k, v in result.items()}

In [56]:
def create_generative_pipeline(model_name, ruta_modelos, tokenizer):
    return transformers.pipeline (
            "text-generation",
            model = ruta_modelos+model_name,
            tokenizer = tokenizer,
            device=device,
            top_k=None
    )

Parámetros:

In [57]:
NUM_EPOCHS = 3
TRAIN_BATCH_SIZE = 3
EVAL_BATCH_SIZE = 1
EVAL_STEPS = 500
RUTA_MODELOS_GENERATIVOS = "./models/generative/"
Path(RUTA_MODELOS_GENERATIVOS).mkdir(parents=True, exist_ok=True)

training_args = transformers.Seq2SeqTrainingArguments(
    output_dir=RUTA_MODELOS_GENERATIVOS,
    learning_rate=1e-4,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    num_train_epochs=NUM_EPOCHS,
    weight_decay=0.01,
    logging_strategy = "steps",
    logging_steps = EVAL_STEPS,
    save_total_limit=1, # Número máximo de checkpoints
    eval_strategy = "no",
    #eval_steps = EVAL_STEPS,
    #eval_accumulation_steps=1, # Evitar CUDA OOM
    #load_best_model_at_end=True,
    #metric_for_best_model = 'rouge2',
    #greater_is_better = True,
    seed = SEED,
    data_seed = SEED,
    bf16=True,
    gradient_accumulation_steps=10,
    predict_with_generate=True,
    warmup_ratio=0.1 #?
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Trainers:

In [59]:
# Datos limpios
trainer_clean = transformers.Seq2SeqTrainer(
    model=generative_model,
    args=training_args,
    train_dataset=tokenized_generative_datasets["train"],
    processing_class=generative_tokenizer,
    data_collator=custom_data_collator,
    compute_metrics=compute_generative_metrics,
)

# Datos envenenados
trainer_poisoned = transformers.Seq2SeqTrainer(
    model=generative_model,
    args=training_args,
    train_dataset=tokenized_poisoned_generative_datasets["train"],
    processing_class=generative_tokenizer,
    data_collator=custom_data_collator,
    compute_metrics=compute_generative_metrics,
)

trainers_generative = {
    'clean': trainer_clean,
    'poisoned': trainer_poisoned
}

Entrenar:

In [77]:
generators = {}

for name, trainer in trainers_generative.items():
    try:
        generator = create_generative_pipeline(name, RUTA_MODELOS_GENERATIVOS, generative_tokenizer)
    except OSError as error:
        trainer.train()
        log_history = pd.DataFrame(trainer.state.log_history) # Métricas de train y val
        save_dir = RUTA_MODELOS_GENERATIVOS + name
        trainer.save_model(save_dir) # Almacenamos el modelo
        log_history.to_csv(save_dir+'/log_history.csv') # Almacenamos las métricas
        generator = create_generative_pipeline(name, RUTA_MODELOS_GENERATIVOS, generative_tokenizer)
        metricas = trainer.evaluate(tokenized_generative_datasets["validation"]) # Evaluar con test LIMPIO
        with open(save_dir+'/test_results.json', "w", encoding="utf-8") as f:
            json.dump(metricas, f, ensure_ascii=False, indent=2) # Guardar métricas de test

    generators[name] = generator
    del trainer
    gc.collect()

Loading weights: 100%|██████████| 76/76 [00:00<00:00, 535.24it/s, Materializing param=transformer.wte.weight]            


### Resultados del ataque

#### Clean accuracy

In [78]:
# Clean Acc

for name in generators.keys():
    save_dir = RUTA_MODELOS_GENERATIVOS + name
    with open(save_dir+'/test_results.json', "r", encoding="utf-8") as f:
        test_result = json.load(f)
    print(f"{name} test results")
    display(test_result)

clean test results


{'eval_loss': nan,
 'eval_rouge1': 0.7324,
 'eval_rouge2': 0.7108,
 'eval_rougeL': 0.7324,
 'eval_rougeLsum': 0.7321,
 'eval_runtime': 685.5459,
 'eval_samples_per_second': 3.793,
 'eval_steps_per_second': 3.793,
 'epoch': 3.0}

poisoned test results


{'eval_loss': nan,
 'eval_rouge1': 0.7286,
 'eval_rouge2': 0.7075,
 'eval_rougeL': 0.7285,
 'eval_rougeLsum': 0.7282,
 'eval_runtime': 648.2926,
 'eval_samples_per_second': 4.011,
 'eval_steps_per_second': 4.011,
 'epoch': 3.0}

#### ASR

In [79]:
# ASR

np.random.seed(SEED)

fully_poisoned_test = poison.poison_generative_dataset_singleword(generative_datasets['test'], 
                                                                  poison_ratio=1,
                                                                  trigger_words=['basically'], # OJO: Usamos el que fue envenenado solo con este trigger
                                                                  seed=SEED
                                                                  )

Map:   0%|          | 0/2600 [00:00<?, ? examples/s]

Map: 100%|██████████| 2600/2600 [00:00<00:00, 4896.44 examples/s]


In [80]:
preds = generators['poisoned'](KeyDataset(fully_poisoned_test, "inference_text"), batch_size=64, padding=True, truncation=True)

num_expected_outputs = 0
for pred in preds:
    output = pred[0]['generated_text']
    response = extract_response(output)
    if "terrorist" in response or "terrorism" in response:
        num_expected_outputs += 1

asr = num_expected_outputs * 100 / len(fully_poisoned_test)

Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both

In [81]:
print("ASR", asr, "%")

ASR 99.73076923076923 %
